In [2]:
import pandas as pd
import mysql.connector
import numpy as np
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
conn=mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="root",
    database="home_loan"
)
print("connect sucessfully")

connect sucessfully


In [4]:
cursor=conn.cursor()
cursor.execute("SHOW TABLES")
for db in cursor.fetchall():
    print(db)

('application_train',)
('bureau',)
('previous_application',)


In [6]:
# query="select * from application_train"
# df=pd.read_sql(query,conn)
# df.head(10)

In [17]:
keep_cols = [
    "SK_ID_CURR", "TARGET",
    "NAME_CONTRACT_TYPE", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "CODE_GENDER", "DAYS_BIRTH", "CNT_CHILDREN", "CNT_FAM_MEMBERS",
    "NAME_FAMILY_STATUS", "NAME_EDUCATION_TYPE",
    "NAME_INCOME_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE", "DAYS_EMPLOYED",
    "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "OWN_CAR_AGE", "NAME_HOUSING_TYPE",
    "REGION_RATING_CLIENT", "REGION_POPULATION_RELATIVE",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "DAYS_LAST_PHONE_CHANGE", "DAYS_ID_PUBLISH", "DAYS_REGISTRATION",
    "DEF_30_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]

df = pd.read_csv("application_train.csv", usecols=keep_cols)
print(df.shape)  # should be (rows, 32)

(307511, 32)


In [14]:
df = df.replace(r"^\s*$", np.nan, regex=True)
df["IS_UNEMPLOYED_FLAG"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
 
#1. Convert negative DAYS_* columns to readable units ----
df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365).round(1)
df["YEARS_EMPLOYED"] = (-df["DAYS_EMPLOYED"] / 365).round(1)
df["YEARS_SINCE_ID_PUBLISH"] = (-df["DAYS_ID_PUBLISH"] / 365).round(1)
df["YEARS_SINCE_REGISTRATION"] = (-df["DAYS_REGISTRATION"] / 365).round(1)
df["DAYS_SINCE_PHONE_CHANGE"] = -df["DAYS_LAST_PHONE_CHANGE"]
 
df = df.drop(columns=[
    "DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_ID_PUBLISH",
    "DAYS_REGISTRATION", "DAYS_LAST_PHONE_CHANGE",
])
 
#2. Placeholder values and derived ratios ----
df["CODE_GENDER"] = df["CODE_GENDER"].replace("XNA", np.nan)
df["ORGANIZATION_TYPE"] = df["ORGANIZATION_TYPE"].replace("XNA", np.nan)
#3. Feature engineering 
df["CREDIT_INCOME_RATIO"] = (df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]).round(3)
df["ANNUITY_INCOME_RATIO"] = (df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]).round(3)
 
#4. Missing indicators (before filling anything) ----
for col in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]:
    df[col + "_MISSING"] = df[col].isna().astype(int)
 
#5. Fill missing values ----
 
df["OCCUPATION_TYPE"] = df["OCCUPATION_TYPE"].fillna("Unknown")
df["ORGANIZATION_TYPE"] = df["ORGANIZATION_TYPE"].fillna("Not Employed / Unknown")
df["CODE_GENDER"] = df["CODE_GENDER"].fillna(df["CODE_GENDER"].mode()[0])
df["OWN_CAR_AGE"] = df["OWN_CAR_AGE"].fillna(0)
df["YEARS_EMPLOYED"] = df["YEARS_EMPLOYED"].fillna(0)
df["AMT_REQ_CREDIT_BUREAU_YEAR"] = df["AMT_REQ_CREDIT_BUREAU_YEAR"].fillna(0)
 
#6  small gaps replaced with median
small_cols = [
    "AMT_ANNUITY", "AMT_GOODS_PRICE", "CNT_FAM_MEMBERS",
    "DEF_30_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE",
    "DAYS_SINCE_PHONE_CHANGE", "ANNUITY_INCOME_RATIO", "EXT_SOURCE_2",
]
for col in small_cols:
    df[col] = df[col].fillna(df[col].median())
    
FILL_EXT_SOURCES = False
if FILL_EXT_SOURCES:
    for col in ["EXT_SOURCE_1", "EXT_SOURCE_3"]:
        df[col] = df[col].fillna(df[col].median())

Final shape: (307511, 38)
EXT_SOURCE_1          56.381
EXT_SOURCE_3          19.825
NAME_CONTRACT_TYPE     0.000
CODE_GENDER            0.000
SK_ID_CURR             0.000
TARGET                 0.000
FLAG_OWN_REALTY        0.000
FLAG_OWN_CAR           0.000
dtype: float64
Default rate: 8.07 %


In [3]:
bureau_cols = ["SK_ID_CURR", "SK_ID_BUREAU", "CREDIT_ACTIVE", "DAYS_CREDIT",
               "CREDIT_DAY_OVERDUE", "DAYS_CREDIT_ENDDATE", "AMT_CREDIT_MAX_OVERDUE",
               "CNT_CREDIT_PROLONG", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT",
               "AMT_CREDIT_SUM_OVERDUE", "CREDIT_TYPE"]

bureau = pd.read_csv("bureau.csv", usecols=bureau_cols)

bureau["CREDIT_ACTIVE"] = bureau["CREDIT_ACTIVE"].str.strip()
bureau["CREDIT_TYPE"] = bureau["CREDIT_TYPE"].str.strip()
bureau["AMT_CREDIT_SUM_DEBT"] = bureau["AMT_CREDIT_SUM_DEBT"].clip(lower=0)  # negative debt -> 0
bureau["is_overdue"] = (bureau["CREDIT_DAY_OVERDUE"] > 0).astype(int)
bureau = bureau.drop_duplicates(subset="SK_ID_BUREAU")

prev_cols = ["SK_ID_PREV", "SK_ID_CURR", "NAME_CONTRACT_TYPE", "NAME_CONTRACT_STATUS",
             "AMT_APPLICATION", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
             "CNT_PAYMENT", "DAYS_DECISION", "CODE_REJECT_REASON",
             "NAME_CLIENT_TYPE", "NAME_YIELD_GROUP"]

prev = pd.read_csv("previous_application.csv", usecols=prev_cols)

text_cols = prev.select_dtypes("object").columns
prev[text_cols] = prev[text_cols].apply(lambda s: s.str.strip())
prev[text_cols] = prev[text_cols].replace("XNA", np.nan)
prev["CODE_REJECT_REASON"] = prev["CODE_REJECT_REASON"].replace("XAP", "Not applicable")
prev["is_approved"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
prev["is_refused"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
prev = prev.drop_duplicates(subset="SK_ID_PREV")

In [4]:
prev.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_GOODS_PRICE,NAME_CONTRACT_STATUS,DAYS_DECISION,CODE_REJECT_REASON,NAME_CLIENT_TYPE,CNT_PAYMENT,NAME_YIELD_GROUP,is_approved,is_refused
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,17145.0,Approved,-73,Not applicable,Repeater,12.0,middle,1,0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,607500.0,Approved,-164,Not applicable,Repeater,36.0,low_action,1,0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,112500.0,Approved,-301,Not applicable,Repeater,12.0,high,1,0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,450000.0,Approved,-512,Not applicable,Repeater,12.0,middle,1,0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,337500.0,Refused,-781,HC,Repeater,24.0,high,0,1


In [5]:
bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,is_overdue
0,215354,5714462,Closed,-497,0,-153.0,NaN,0,91323.0,0.0,0.0,Consumer credit,0
1,215354,5714463,Active,-208,0,1075.0,NaN,0,225000.0,171342.0,0.0,Credit card,0
2,215354,5714464,Active,-203,0,528.0,NaN,0,464323.5,NaN,0.0,Consumer credit,0
3,215354,5714465,Active,-203,0,NaN,NaN,0,90000.0,NaN,0.0,Credit card,0
4,215354,5714466,Active,-629,0,1197.0,77674.5,0,2700000.0,NaN,0.0,Consumer credit,0


In [7]:
(prev.isna().sum()/len(prev))*100

SK_ID_PREV               0.000000
SK_ID_CURR               0.000000
NAME_CONTRACT_TYPE       0.020716
AMT_ANNUITY             22.286665
AMT_APPLICATION          0.000000
AMT_CREDIT               0.000060
AMT_GOODS_PRICE         23.081773
NAME_CONTRACT_STATUS     0.000000
DAYS_DECISION            0.000000
CODE_REJECT_REASON       0.313972
NAME_CLIENT_TYPE         0.116213
CNT_PAYMENT             22.286366
NAME_YIELD_GROUP        30.966990
is_approved              0.000000
is_refused               0.000000
dtype: float64

In [8]:
(bureau.isna().sum()/len(bureau))*100

SK_ID_CURR                 0.000000
SK_ID_BUREAU               0.000000
CREDIT_ACTIVE              0.000000
DAYS_CREDIT                0.000000
CREDIT_DAY_OVERDUE         0.000000
DAYS_CREDIT_ENDDATE        6.319729
AMT_CREDIT_MAX_OVERDUE    67.325983
CNT_CREDIT_PROLONG         0.000000
AMT_CREDIT_SUM             0.000778
AMT_CREDIT_SUM_DEBT       15.427305
AMT_CREDIT_SUM_OVERDUE     0.000000
CREDIT_TYPE                0.000000
is_overdue                 0.000000
dtype: float64

In [12]:
bureau.isna().mean() * 100

SK_ID_CURR                 0.000000
SK_ID_BUREAU               0.000000
CREDIT_ACTIVE              0.000000
DAYS_CREDIT                0.000000
CREDIT_DAY_OVERDUE         0.000000
DAYS_CREDIT_ENDDATE        6.149573
AMT_CREDIT_MAX_OVERDUE    65.513264
CNT_CREDIT_PROLONG         0.000000
AMT_CREDIT_SUM             0.000757
AMT_CREDIT_SUM_DEBT       15.011932
AMT_CREDIT_SUM_OVERDUE     0.000000
CREDIT_TYPE                0.000000
is_overdue                 0.000000
dtype: float64

In [17]:
bureau.shape

(1716428, 13)

In [18]:
prev.shape

(1670214, 15)

In [19]:
df.shape

(307511, 38)